# Phần mở rộng mục 2 (không nộp)

- Cell 2.5–2.10 cũ của `llm_presentation.ipynb`: mBERT vs Qwen, phép thử đổi token cuối, VLM, LSTM vs Transformer.
- Tách ra ngày 19/11 theo file chốt khung v4 §5: chỉ làm khi phần chính xong; số liệu chính thức không phụ thuộc file này.
- **Không chạy độc lập được**: cần biến của phần 0 và cell 2.1–2.4 (`model`, `tok`, `PART02_RECORDS`, `top_k_next_tokens`...). Muốn chạy thì chép các cell này vào sau cell 2.4 của notebook chính.
- Output giữ nguyên từ smoke run `20261117_161656_smoke`.

### 2.5 Demo A (1/3): mBERT điền chỗ trống, nhìn cả hai phía
- **Mục đích:** so một model **encoder-only** (mBERT, nhìn hai chiều) với Qwen (decoder-only, chỉ nhìn quá khứ). mBERT đoán token bị che `[MASK]` và dùng được cả phần **bên phải** (`là Paris.`).
- **Đầu vào:** `MODEL_MBERT`; câu chia 3 phần `DEMO_A` (bên trái, đáp án, bên phải). Đáp án phải là **1 token** của mBERT, vì `[MASK]` chỉ thay được một token (kiểm bằng `assert`).
- **Đầu ra:** `fill` (pipeline fill-mask, giữ lại cho 2.7); `bert_df` (top-5 + xác suất).

In [55]:
# 2.5 mBERT: pipeline fill-mask đoán [MASK] bằng ngữ cảnh hai phía
from transformers import pipeline

MODEL_MBERT = "google-bert/bert-base-multilingual-cased"
DEMO_A = {"left": "Thủ đô của", "answer": "Pháp", "right": "là Paris."}

fill = pipeline("fill-mask", model=MODEL_MBERT, device=model.device)  # cùng GPU với model chính
fill.model.eval()
bert_tok = fill.tokenizer
n_mbert = sum(p.numel() for p in fill.model.parameters())
bert_text = f"{DEMO_A['left']} {bert_tok.mask_token} {DEMO_A['right']}"
assert len(bert_tok.tokenize(DEMO_A["answer"])) == 1, bert_tok.tokenize(DEMO_A["answer"])  # đáp án là 1 token

cuda_sync()
t0 = time.perf_counter()
bert_pred = fill(bert_text, top_k=5)  # list 5 dict: token_str, score
cuda_sync()
bert_sec = time.perf_counter() - t0
bert_df = pd.DataFrame({"rank": range(1, 6), "token": [p["token_str"] for p in bert_pred],
                        "prob": [p["score"] for p in bert_pred]})
add_topk_records("mbert", bert_text, bert_tok.mask_token, bert_df, bert_sec)
# Xác suất mBERT gán cho đáp án (dù đáp án không lọt top-5)
bert_answer_prob = fill(bert_text, targets=[DEMO_A["answer"]])[0]["score"]

log(f"mBERT: {MODEL_MBERT} ({n_mbert:,} tham số) | input: {bert_text!r} | "
    f"P('{DEMO_A['answer']}') = {bert_answer_prob:.4f}", part=PART02)
print(f"INPUT mBERT (thấy cả hai phía): {bert_text!r}")
print(bert_df.to_string(index=False))
print(f"P({DEMO_A['answer']!r}) = {bert_answer_prob:.4f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mBERT: google-bert/bert-base-multilingual-cased (177,974,523 tham số) | input: 'Thủ đô của [MASK] là Paris.' | P('Pháp') = 0.0558
INPUT mBERT (thấy cả hai phía): 'Thủ đô của [MASK] là Paris.'
 rank token     prob
    1    nó 0.747069
    2  Pháp 0.055787
    3  tỉnh 0.037385
    4   phủ 0.010666
    5  quận 0.009056
P('Pháp') = 0.0558


### 2.6 Demo A (2/3): Qwen chỉ thấy phần bên trái
- **Mục đích:** cùng câu, Qwen chỉ nhận phần **trước** chỗ trống (`Thủ đô của`) và đoán token tiếp theo. Qwen không thấy `là Paris.`, vì decoder-only sinh từ trái sang phải.
- **Đầu vào:** `DEMO_A`; `top_k_next_tokens()` (2.4).
- **Đầu ra:** `RESULTS_DIR/compare_bert_qwen.csv` (`model, input_seen, rank, token, prob`; thêm một dòng `rank = "đáp án"` cho xác suất của đáp án).

In [56]:
# 2.6 Qwen: chỉ đưa phần bên trái, so với mBERT
cuda_sync()
t0 = time.perf_counter()
qwen_a_df, qwen_a_ids = top_k_next_tokens(model, tok, DEMO_A["left"], k=5)
cuda_sync()
add_topk_records("qwen", DEMO_A["left"], tok.decode([qwen_a_ids[-1]]), qwen_a_df, time.perf_counter() - t0)

# Xác suất Qwen gán cho đáp án: token đầu tiên của " Pháp" (có dấu cách, vì đứng sau "của")
answer_ids_qwen = tok(" " + DEMO_A["answer"], add_special_tokens=False)["input_ids"]
with torch.no_grad():
    logits_a = model(torch.tensor([qwen_a_ids], device=model.device)).logits[0, -1].float()
qwen_answer_prob = torch.softmax(logits_a, dim=-1)[answer_ids_qwen[0]].item()

rows = []
for name, seen, df, p_ans in [("mBERT (encoder-only)", bert_text, bert_df, bert_answer_prob),
                              ("Qwen2.5-1.5B-Instruct (decoder-only)", DEMO_A["left"], qwen_a_df, qwen_answer_prob)]:
    rows += [{"model": name, "input_seen": seen, "rank": r["rank"], "token": r["token"], "prob": r["prob"]}
             for _, r in df.iterrows()]
    rows.append({"model": name, "input_seen": seen, "rank": "đáp án", "token": DEMO_A["answer"], "prob": p_ans})
compare_a_df = pd.DataFrame(rows)
compare_a_df.to_csv(RESULTS_DIR / "compare_bert_qwen.csv", index=False, encoding="utf-8")
print(f"INPUT Qwen (chỉ thấy bên trái): {DEMO_A['left']!r}")
print(compare_a_df.to_string(index=False))

log(f"Qwen: input {DEMO_A['left']!r} | P(' {DEMO_A['answer']}', token đầu, {len(answer_ids_qwen)} token) = "
    f"{qwen_answer_prob:.4f} | mBERT P = {bert_answer_prob:.4f}", part=PART02)

# Tự kiểm: đủ 2 model × (5 + 1) dòng
assert len(compare_a_df) == 12

INPUT Qwen (chỉ thấy bên trái): 'Thủ đô của'
                               model                  input_seen   rank   token     prob
                mBERT (encoder-only) Thủ đô của [MASK] là Paris.      1      nó 0.747069
                mBERT (encoder-only) Thủ đô của [MASK] là Paris.      2    Pháp 0.055787
                mBERT (encoder-only) Thủ đô của [MASK] là Paris.      3    tỉnh 0.037385
                mBERT (encoder-only) Thủ đô của [MASK] là Paris.      4     phủ 0.010666
                mBERT (encoder-only) Thủ đô của [MASK] là Paris.      5    quận 0.009056
                mBERT (encoder-only) Thủ đô của [MASK] là Paris. đáp án    Pháp 0.055787
Qwen2.5-1.5B-Instruct (decoder-only)                  Thủ đô của      1    Việt 0.103773
Qwen2.5-1.5B-Instruct (decoder-only)                  Thủ đô của      2       � 0.037002
Qwen2.5-1.5B-Instruct (decoder-only)                  Thủ đô của      3    nước 0.035863
Qwen2.5-1.5B-Instruct (decoder-only)                  Thủ đô của 

### 2.7 Demo A (3/3): đổi token cuối, token phía trước có đổi không
- **Mục đích:** đo trực tiếp "chỉ nhìn quá khứ". Đưa 2 câu chỉ khác **token cuối** (`đẹp` / `tối`) vào cả hai model, rồi so hidden state ở lớp cuối tại **các vị trí phía trước**:
  - Qwen (causal): các vị trí trước không nhìn thấy token cuối, nên hidden state **không đổi**;
  - mBERT (hai chiều): mọi vị trí đều nhìn token cuối, nên hidden state **đổi**.
- **Đầu vào:** `SWAP_TEXTS`; `model`, `tok`; `fill.model`, `bert_tok` (2.5). Hai từ được thay phải có cùng số token ở cả hai tokenizer (kiểm bằng `assert`).
- **Đầu ra:** `RESULTS_DIR/compare_causal_swap.csv` (`model, position, token, max_abs_diff, is_changed_token`).
- So bằng `torch.allclose` có ngưỡng sai số (chạy fp16). Chỉ `assert` phía Qwen, vì đó là tính chất toán học của causal mask. Phía mBERT ghi đúng số đo, không `assert`.

In [57]:
# 2.7 Phép thử đổi token cuối: so hidden state lớp cuối ở các vị trí phía trước
SWAP_TEXTS = ("Hôm nay trời rất đẹp", "Hôm nay trời rất tối")  # "đẹp", "tối": 1 token ở cả Qwen và mBERT


def last_hidden(mdl, tokenizer, text: str, add_special_tokens: bool) -> tuple[list[int], torch.Tensor]:
    """Token ID và hidden state lớp cuối (n_token × hidden, float32) của một câu."""
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=add_special_tokens).to(mdl.device)
    with torch.no_grad():
        out = mdl(**enc, output_hidden_states=True)
    return enc["input_ids"][0].tolist(), out.hidden_states[-1][0].float()


swap_rows = []
SWAP_SUMMARY = {}
# Qwen không thêm token đặc biệt; mBERT thêm [CLS] ở đầu, [SEP] ở cuối (giữ, vì đó là cách mBERT được dùng)
for name, mdl, tk, special in [("Qwen2.5-1.5B-Instruct", model, tok, False), ("mBERT", fill.model, bert_tok, True)]:
    ids_a, h_a = last_hidden(mdl, tk, SWAP_TEXTS[0], special)
    ids_b, h_b = last_hidden(mdl, tk, SWAP_TEXTS[1], special)
    changed = [i for i, (x, y) in enumerate(zip(ids_a, ids_b)) if x != y]
    assert len(ids_a) == len(ids_b) and len(changed) == 1, (name, ids_a, ids_b)  # chỉ khác đúng 1 token
    pos = changed[0]
    diff = (h_a - h_b).abs().max(dim=-1).values  # mỗi vị trí: chênh lệch lớn nhất trên 1,536 (768) chiều
    for i, t in enumerate(ids_a):
        swap_rows.append({"model": name, "position": i, "token": tk.decode([t]),
                          "max_abs_diff": round(diff[i].item(), 6), "is_changed_token": i == pos})
    before_same = torch.allclose(h_a[:pos], h_b[:pos], atol=1e-3, rtol=1e-3)
    SWAP_SUMMARY[name] = {"changed_position": pos, "max_diff_before": diff[:pos].max().item(),
                          "prefix_unchanged": before_same}
    log(f"đổi token cuối | {name}: vị trí đổi {pos}, chênh lệch lớn nhất ở các vị trí trước = "
        f"{diff[:pos].max().item():.6f} → {'không đổi' if before_same else 'ĐỔI'}", part=PART02)

swap_df = pd.DataFrame(swap_rows)
swap_df.to_csv(RESULTS_DIR / "compare_causal_swap.csv", index=False, encoding="utf-8")
print(f"Câu A: {SWAP_TEXTS[0]!r} | câu B: {SWAP_TEXTS[1]!r}")
print(swap_df.to_string(index=False))

# Tự kiểm: causal mask bảo đảm hidden state của Qwen ở các vị trí trước token bị đổi không đổi
assert SWAP_SUMMARY["Qwen2.5-1.5B-Instruct"]["prefix_unchanged"], SWAP_SUMMARY

đổi token cuối | Qwen2.5-1.5B-Instruct: vị trí đổi 6, chênh lệch lớn nhất ở các vị trí trước = 0.000000 → không đổi
đổi token cuối | mBERT: vị trí đổi 7, chênh lệch lớn nhất ở các vị trí trước = 0.566341 → ĐỔI
Câu A: 'Hôm nay trời rất đẹp' | câu B: 'Hôm nay trời rất tối'
                model  position token  max_abs_diff  is_changed_token
Qwen2.5-1.5B-Instruct         0     H      0.000000             False
Qwen2.5-1.5B-Instruct         1    ôm      0.000000             False
Qwen2.5-1.5B-Instruct         2     n      0.000000             False
Qwen2.5-1.5B-Instruct         3    ay      0.000000             False
Qwen2.5-1.5B-Instruct         4  trời      0.000000             False
Qwen2.5-1.5B-Instruct         5   rất      0.000000             False
Qwen2.5-1.5B-Instruct         6   đẹp     18.031250              True
                mBERT         0 [CLS]      0.256013             False
                mBERT         1     H      0.345848             False
                mBERT       

### 2.8 LLM so với VLM: cấu trúc Qwen2-VL-2B [cần chốt]
- **Mục đích:** cho thấy VLM = **bộ mã hoá ảnh + LLM decoder-only**. Đọc config của `Qwen/Qwen2-VL-2B-Instruct`, dựng khung model trên thiết bị `meta` (chỉ có shape, **không tải trọng số**, không tốn VRAM), rồi đếm tham số theo thành phần.
- **Đầu vào:** `MODEL_VLM` (chỉ tải `config.json`); `ARCH` (cell 0.5) để so.
- **Đầu ra:** `RESULTS_DIR/compare_vlm.csv` (`item, qwen2.5_1.5b_instruct, qwen2_vl_2b`).
- Phương án (a) của Toàn (`tra_loi_toan_cho_thu_v1.md` §D), **chờ Thư chốt**. Không chạy model VLM trên ảnh.
- Lưu ý khi nói: phần LLM của Qwen2-VL là **thế hệ Qwen2**, không phải Qwen2.5. Cùng cấu hình, khác trọng số và cách huấn luyện.

In [58]:
# 2.8 VLM: dựng khung Qwen2-VL-2B trên "meta" (không tải trọng số), đếm tham số theo thành phần
from transformers import AutoConfig, Qwen2VLForConditionalGeneration

MODEL_VLM = "Qwen/Qwen2-VL-2B-Instruct"
vlm_cfg = AutoConfig.from_pretrained(MODEL_VLM)  # chỉ tải config.json (vài KB)
with torch.device("meta"):  # tensor trên "meta" chỉ có shape, không có giá trị, không chiếm bộ nhớ
    vlm_skeleton = Qwen2VLForConditionalGeneration(vlm_cfg)

vlm_groups = {"bộ mã hoá ảnh (visual)": 0, "LLM decoder-only": 0}
for name, p in vlm_skeleton.named_parameters():
    vlm_groups["bộ mã hoá ảnh (visual)" if "visual" in name else "LLM decoder-only"] += p.numel()
n_vlm = sum(vlm_groups.values())

txt = vlm_cfg.text_config  # cấu hình phần LLM bên trong VLM
vis = vlm_cfg.vision_config
CFG_KEYS = ["num_hidden_layers", "hidden_size", "num_attention_heads", "num_key_value_heads",
            "intermediate_size", "vocab_size"]
vlm_rows = [{"item": k, "qwen2.5_1.5b_instruct": ARCH[k], "qwen2_vl_2b": getattr(txt, k)} for k in CFG_KEYS]
vlm_rows += [
    {"item": "n_params LLM", "qwen2.5_1.5b_instruct": ARCH["n_params"], "qwen2_vl_2b": vlm_groups["LLM decoder-only"]},
    {"item": "n_params bộ mã hoá ảnh", "qwen2.5_1.5b_instruct": 0, "qwen2_vl_2b": vlm_groups["bộ mã hoá ảnh (visual)"]},
    {"item": "n_params tổng", "qwen2.5_1.5b_instruct": ARCH["n_params"], "qwen2_vl_2b": n_vlm},
    {"item": "visual: số block ViT", "qwen2.5_1.5b_instruct": "—", "qwen2_vl_2b": vis.depth},
    {"item": "visual: patch_size", "qwen2.5_1.5b_instruct": "—", "qwen2_vl_2b": vis.patch_size},
    # visual.hidden_size = số chiều sau lớp nối (merger): phải bằng hidden của LLM để ghép token ảnh vào chuỗi
    {"item": "visual: số chiều đầu ra", "qwen2.5_1.5b_instruct": "—", "qwen2_vl_2b": vis.hidden_size},
]
vlm_df = pd.DataFrame(vlm_rows)
vlm_df.to_csv(RESULTS_DIR / "compare_vlm.csv", index=False, encoding="utf-8")
print(vlm_df.to_string(index=False))
log(f"VLM {MODEL_VLM}: {n_vlm:,} tham số = visual {vlm_groups['bộ mã hoá ảnh (visual)']:,} + "
    f"LLM {vlm_groups['LLM decoder-only']:,} (khung trên meta, không tải trọng số)", part=PART02)

# Tự kiểm: tổng khớp số trên Hugging Face (gop_y_thu_cho_toan_v1.md §4.2); token ảnh có cùng số chiều với LLM
assert n_vlm == 2_208_985_600, n_vlm
assert vis.hidden_size == txt.hidden_size
del vlm_skeleton

                   item qwen2.5_1.5b_instruct  qwen2_vl_2b
      num_hidden_layers                    28           28
            hidden_size                  1536         1536
    num_attention_heads                    12           12
    num_key_value_heads                     2            2
      intermediate_size                  8960         8960
             vocab_size                151936       151936
           n_params LLM            1543714304   1543714304
 n_params bộ mã hoá ảnh                     0    665271296
          n_params tổng            1543714304   2208985600
   visual: số block ViT                     —           32
     visual: patch_size                     —           14
visual: số chiều đầu ra                     —         1536
VLM Qwen/Qwen2-VL-2B-Instruct: 2,208,985,600 tham số = visual 665,271,296 + LLM 1,543,714,304 (khung trên meta, không tải trọng số)


### 2.9 RNN/LSTM so với Transformer: tuần tự hay song song [cần chốt]
- **Mục đích:** minh hoạ **cách tính**, không nói về chất lượng. LSTM phải đi qua chuỗi từng bước (bước t cần hidden state của bước t−1). Một lớp Transformer có causal mask tính mọi vị trí cùng lúc. Đo thời gian chạy hết chuỗi dài 128 / 512 / 1,024 token.
- **Đầu vào:** `nn.LSTM` và `nn.TransformerEncoderLayer` cùng số chiều `RNN_D`, **trọng số ngẫu nhiên** (không train), dữ liệu ngẫu nhiên.
- **Đầu ra:** `RESULTS_DIR/compare_rnn_transformer.csv` (`model, seq_len, n_params, sequential_steps, ms_mean`).
- Đề xuất của Toàn (`tra_loi_toan_cho_thu_v1.md` §D), **chờ Thư chốt**. LSTM trên GPU dùng cuDNN đã tối ưu; kết quả ghi đúng như đo, kể cả khi LSTM không chậm hơn.

In [59]:
# 2.9 LSTM (tuần tự) so với 1 lớp Transformer có causal mask (song song), trọng số ngẫu nhiên
import torch.nn as nn

RNN_D = 512                        # số chiều vector mỗi bước
RNN_SEQ_LENS = [128, 512, 1024]    # độ dài chuỗi
RNN_REPEAT = 3 if SMOKE else 20    # số lần đo, lấy trung bình

torch.manual_seed(SEED)
lstm = nn.LSTM(RNN_D, RNN_D, batch_first=True).to(model.device).eval()
tf_layer = nn.TransformerEncoderLayer(RNN_D, nhead=8, dim_feedforward=4 * RNN_D, batch_first=True).to(model.device).eval()


def time_ms(fn, repeat: int) -> float:
    """Thời gian trung bình (ms) của fn(); chạy 2 lần khởi động trước khi đo."""
    with torch.no_grad():
        for _ in range(2):
            fn()
        cuda_sync()
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn()
        cuda_sync()
    return (time.perf_counter() - t0) / repeat * 1000


rnn_rows = []
for L in RNN_SEQ_LENS:
    x = torch.randn(1, L, RNN_D, device=model.device)
    mask = nn.Transformer.generate_square_subsequent_mask(L, device=model.device)  # −∞ ở vị trí tương lai
    rnn_rows.append({"model": "LSTM", "seq_len": L, "n_params": sum(p.numel() for p in lstm.parameters()),
                     "sequential_steps": L, "ms_mean": round(time_ms(lambda: lstm(x), RNN_REPEAT), 3)})
    rnn_rows.append({"model": "Transformer layer (causal)", "seq_len": L,
                     "n_params": sum(p.numel() for p in tf_layer.parameters()), "sequential_steps": 1,
                     "ms_mean": round(time_ms(lambda: tf_layer(x, src_mask=mask, is_causal=True), RNN_REPEAT), 3)})

rnn_df = pd.DataFrame(rnn_rows)
rnn_df.to_csv(RESULTS_DIR / "compare_rnn_transformer.csv", index=False, encoding="utf-8")
print(rnn_df.to_string(index=False))
log(f"LSTM vs Transformer (d={RNN_D}, repeat={RNN_REPEAT}, {model.device}): "
    f"{rnn_df[['model', 'seq_len', 'ms_mean']].to_dict('records')}", part=PART02)

# Tự kiểm: đủ 2 model × số độ dài; mọi thời gian dương
assert len(rnn_df) == 2 * len(RNN_SEQ_LENS) and (rnn_df["ms_mean"] > 0).all()
del lstm, tf_layer, x, mask

                     model  seq_len  n_params  sequential_steps  ms_mean
                      LSTM      128   2101248               128    2.044
Transformer layer (causal)      128   3152384                 1    0.476
                      LSTM      512   2101248               512    8.552
Transformer layer (causal)      512   3152384                 1    1.313
                      LSTM     1024   2101248              1024   16.628
Transformer layer (causal)     1024   3152384                 1    2.811
LSTM vs Transformer (d=512, repeat=3, cuda:0): [{'model': 'LSTM', 'seq_len': 128, 'ms_mean': 2.044}, {'model': 'Transformer layer (causal)', 'seq_len': 128, 'ms_mean': 0.476}, {'model': 'LSTM', 'seq_len': 512, 'ms_mean': 8.552}, {'model': 'Transformer layer (causal)', 'seq_len': 512, 'ms_mean': 1.313}, {'model': 'LSTM', 'seq_len': 1024, 'ms_mean': 16.628}, {'model': 'Transformer layer (causal)', 'seq_len': 1024, 'ms_mean': 2.811}]


### 2.10 Giải phóng mBERT và tổng kết phần 2
- **Mục đích:** xoá mBERT để nhường bộ nhớ cho phần 5 (bản base); ghi log mỗi mẫu và dòng tổng kết.
- **Đầu vào:** `PART02_RECORDS`.
- **Đầu ra:** `LOG_DIR/part02_arch_records.csv`; dòng END trong `part02_arch.log`; một dòng tóm tắt trong `run.log`.

In [60]:
# 2.10 Tổng kết phần 2
del fill, bert_tok
print(f"VRAM sau khi xoá mBERT: {free_vram()} GB")

pd.DataFrame(PART02_RECORDS, columns=PART02_REC_COLS).to_csv(
    LOG_DIR / f"{PART02}_records.csv", index=False, encoding="utf-8")

PART02_FILES = ["model_config.json", "top5_next_tokens.csv", "compare_bert_qwen.csv",
                "compare_causal_swap.csv", "compare_vlm.csv", "compare_rnn_transformer.csv"]
PART02_FIGS = ["causal_mask.png", "attention_qwen.png"]
for f in PART02_FILES:
    assert (RESULTS_DIR / f).exists(), f
for f in PART02_FIGS:
    assert (FIGURES_DIR / f).exists(), f

log(f"=== PART 02 arch END {now_iso()} ===", part=PART02)
log(f"tổng kết: top-1 sau {TOPK_TEXT!r} = {top5_df['token'].iloc[0]!r} ({top5_df['prob'].iloc[0]:.3f}) | "
    f"P(đáp án) mBERT {bert_answer_prob:.3f} / Qwen {qwen_answer_prob:.3f} | "
    f"đổi token cuối, các vị trí trước: Qwen không đổi = {SWAP_SUMMARY['Qwen2.5-1.5B-Instruct']['prefix_unchanged']}, "
    f"mBERT không đổi = {SWAP_SUMMARY['mBERT']['prefix_unchanged']}", part=PART02)
log(f"[{PART02}] OK {now_iso()} → {', '.join('results/' + f for f in PART02_FILES)}, "
    f"{', '.join('figures/' + f for f in PART02_FIGS)}")

VRAM sau khi xoá mBERT: 9.3 GB
=== PART 02 arch END 2026-11-17T16:17:23+07:00 ===
tổng kết: top-1 sau 'Hôm nay trời rất' = ' đẹp' (0.788) | P(đáp án) mBERT 0.056 / Qwen 0.017 | đổi token cuối, các vị trí trước: Qwen không đổi = True, mBERT không đổi = False
[part02_arch] OK 2026-11-17T16:17:23+07:00 → results/model_config.json, results/top5_next_tokens.csv, results/compare_bert_qwen.csv, results/compare_causal_swap.csv, results/compare_vlm.csv, results/compare_rnn_transformer.csv, figures/causal_mask.png, figures/attention_qwen.png
